## 导入依赖

In [ ]:
import ollama
import json
import pandas as pd
from IPython.display import display, Markdown

## 定义 Prompt 模板变体

In [ ]:
PROMPT_V1 = """
你是一个情感分析专家。请分析以下评论的方面和情感。
输出格式：JSON
"""

PROMPT_V2_COT = """
你是一个精確的方面級別情感分析（ABSA）引擎。
请分析用户提供的评论文本。

步骤：
1. **方面提取**：识别评论中提到的所有关键方面（Aspects）。
2. **情感判斷**：对于每一个方面，判断情感极性 (positive, negative, neutral)。
3. **引用佐證**：提取原文中的关键词句。

请使用 /think 模式展示你的思考过程。
最后仅输出符合以下格式的 JSON：
{
  "aspects": [
    {"aspect_term": "...", "sentiment": "...", "quote": "..."}
  ]
}
"""

## 定义测试函数 (Ollama API)

In [ ]:
def test_prompt_ollama(text, system_prompt, model="qwen3:8b"):
    """
    发送请求给本地 Ollama 服务进行测试
    """
    messages = [
        {"role": "system", "content": system_prompt},
        # 在用户输入后添加 /think 以确保激活思维链（如果是软开关模型）
        {"role": "user", "content": f"{text}\n\n/think"}
    ]
    
    print(f"正在测试模型: {model}...")
    try:
        response = ollama.chat(model=model, messages=messages)
        content = response['message']['content']
        
        # 简单展示
        print("-" * 30)
        print("Raw Output:")
        print(content)
        print("-" * 30)
        
        return content
    except Exception as e:
        print(f"Ollama 调用失败: {e}")
        return None

## 运行测试案例

In [ ]:
# 这里挑选几个具有挑战性的样本（例如包含转折、隐含情感）
test_cases = []

for i, text in enumerate(test_cases):
    display(Markdown(f"### 测试案例 {i+1}: {text}"))
    output = test_prompt_ollama(text, PROMPT_V2_COT)